In [2]:
import pandas as pd

In [4]:
arabic_data = pd.read_csv('data/ar_reviews_100k.tsv', sep='\t')
english_data = pd.read_csv('data/sentiment_data.csv')


In [6]:
arabic_data['language'] = 'Arabic'
english_data['language'] = 'English'

In [13]:
arabic_df = arabic_data[['text', 'language']]
english_df = english_data[['Comment', 'language']].rename(columns={'Comment': 'text'})

In [19]:
arabic_df.isna().sum()
english_df.isna().sum()

text        217
language      0
dtype: int64

In [14]:
combined = pd.concat([arabic_df, english_df])

### Shuffling

In [20]:
combined = combined.sample(frac=1, random_state=42)
combined = combined.dropna(subset=['text'])

### splitting 

In [21]:
X = combined['text']
y = combined['language']

### Vectorization

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
lang_vectorizer = TfidfVectorizer()

X_vectorized = lang_vectorizer.fit_transform(X)

### modeling

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)


In [24]:
from sklearn.naive_bayes import MultinomialNB
lang_model = MultinomialNB()

lang_model.fit(X_train, y_train)
y_pred = lang_model.predict(X_test)

In [25]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      Arabic       1.00      1.00      1.00     20011
     English       1.00      1.00      1.00     48175

    accuracy                           1.00     68186
   macro avg       1.00      1.00      1.00     68186
weighted avg       1.00      1.00      1.00     68186



In [26]:
import pickle

pickle.dump(lang_model, open("lang_model.pkl", "wb"))
pickle.dump(lang_vectorizer, open("lang_vectorizer.pkl", "wb"))